In [0]:
# ============================================================
# GOLD LAYER - CREATE SCHEMA
# ============================================================

spark.sql("""
CREATE SCHEMA IF NOT EXISTS automotive_warranty.gold
""")

print("Gold schema created successfully!")
print("Schema: automotive_warranty.gold")

Gold schema created successfully!
Schema: automotive_warranty.gold


In [0]:
# ============================================================
# GOLD LAYER - CUSTOMER ANALYTICS
# ============================================================

from pyspark.sql.functions import col, count

# Read Silver tables
df_customers = spark.table(
    "automotive_warranty.silver.customers"
)

df_customer_vehicles = spark.table(
    "automotive_warranty.silver.customer_vehicles"
)

# Count vehicles owned by each customer
df_vehicle_count = (
    df_customer_vehicles
    .groupBy("customer_id")
    .agg(
        count("*").alias("vehicle_count")
    )
)

# Join customer information with vehicle count
df_customer_analytics = (
    df_customers
    .join(
        df_vehicle_count,
        on="customer_id",
        how="left"
    )
    .fillna({"vehicle_count": 0})
)

# Write Gold table
df_customer_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.gold.customer_analytics"
    )

print("Gold customer analytics created successfully!")
print(f"Rows: {df_customer_analytics.count():,}")

display(df_customer_analytics.limit(10))

Gold customer analytics created successfully!
Rows: 10,000


customer_id,first_name,last_name,email,phone_number,address_id,created_at,ingestion_timestamp,vehicle_count
11,Vikram,Patel,vikram.patel11@example.com,919815697426,11,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1
13,Rahul,Iyer,rahul.iyer13@example.com,919875976121,13,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1
28,Vikram,Patel,vikram.patel28@example.com,919879224830,28,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1
46,Deepa,Kumar,deepa.kumar46@example.com,919810142681,46,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1
60,Rohan,Rao,rohan.rao60@example.com,919851233970,60,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1
65,Sneha,Verma,sneha.verma65@example.com,919856504843,65,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1
93,Deepa,Patel,deepa.patel93@example.com,919824000185,93,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1
114,Rohan,Reddy,rohan.reddy114@example.com,919831745269,114,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1
116,Karthik,Reddy,karthik.reddy116@example.com,919828496959,116,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1
163,Sneha,Verma,sneha.verma163@example.com,919850619578,163,2024-01-01T10:00:00.000Z,2026-09-04T11:13:18.337Z,1


In [0]:
# ============================================================
# GOLD LAYER - VEHICLE ANALYTICS
# ============================================================

from pyspark.sql.functions import col

# Read Silver tables
df_vehicles = spark.table(
    "automotive_warranty.silver.vehicles"
)

df_vehicle_models = spark.table(
    "automotive_warranty.silver.vehicle_models"
)

# Join vehicles with vehicle model information
df_vehicle_analytics = (
    df_vehicles.alias("v")
    .join(
        df_vehicle_models.alias("m"),
        col("v.model_id") == col("m.model_id"),
        "left"
    )
    .select(
        col("v.vehicle_id"),
        col("v.vin"),
        col("v.engine_number"),
        col("v.model_id"),
        col("m.make"),
        col("m.model_name"),
        col("m.body_type"),
        col("m.fuel_type"),
        col("v.manufacturing_year"),
        col("v.color"),
        col("v.ingestion_timestamp")
    )
)

# Write Gold vehicle analytics table
df_vehicle_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.gold.vehicle_analytics"
    )

print("Gold vehicle analytics created successfully!")
print(f"Rows: {df_vehicle_analytics.count():,}")

display(df_vehicle_analytics.limit(10))

Gold vehicle analytics created successfully!
Rows: 10,000


vehicle_id,vin,engine_number,model_id,make,model_name,body_type,fuel_type,manufacturing_year,color,ingestion_timestamp
1,VIN9IND2024A000001,ENG1000001,5,Hyundai,Creta,Compact SUV,Petrol,2022,Starlight Blue,2026-09-04T11:14:07.791Z
17,VIN9IND2022A000017,ENG1000017,8,Maruti Suzuki,Grand Vitara,Mid SUV,Hybrid,2023,Daytona Grey,2026-09-04T11:14:07.791Z
18,VIN9IND2022A000018,ENG1000018,1,Tata,Nexon EV,Compact SUV,Electric,2023,Phantom Black,2026-09-04T11:14:07.791Z
28,VIN9IND2024A000028,ENG1000028,7,Maruti Suzuki,Brezza,Compact SUV,Hybrid,2021,Flame Red,2026-09-04T11:14:07.791Z
31,VIN9IND2022A000031,ENG1000031,8,Maruti Suzuki,Grand Vitara,Mid SUV,Hybrid,2021,Flame Red,2026-09-04T11:14:07.791Z
32,VIN9IND2023A000032,ENG1000032,9,Toyota,Innova Hycross,MPV,Hybrid,2024,Flame Red,2026-09-04T11:14:07.791Z
34,VIN9IND2023A000034,ENG1000034,2,Tata,Harrier,Mid-size SUV,Diesel,2021,Pearl White,2026-09-04T11:14:07.791Z
44,VIN9IND2024A000044,ENG1000044,7,Maruti Suzuki,Brezza,Compact SUV,Hybrid,2023,Pearl White,2026-09-04T11:14:07.791Z
49,VIN9IND2022A000049,ENG1000049,3,Mahindra,XUV700,Full SUV,Petrol,2024,Phantom Black,2026-09-04T11:14:07.791Z
63,VIN9IND2022A000063,ENG1000063,1,Tata,Nexon EV,Compact SUV,Electric,2023,Pearl White,2026-09-04T11:14:07.791Z


In [0]:
# ============================================================
# GOLD LAYER - SERVICE ANALYTICS
# ============================================================

from pyspark.sql.functions import count, sum, avg, col

# Read Silver service orders
df_service_orders = spark.table(
    "automotive_warranty.silver.service_orders"
)

print("Service Orders columns:")
print(df_service_orders.columns)

# Create basic service metrics
total_service_orders = df_service_orders.count()

# Check for commonly available cost columns
cost_column = None

for c in ["total_cost", "service_cost", "labor_cost", "amount", "cost"]:
    if c in df_service_orders.columns:
        cost_column = c
        break

# Create analytics table
if cost_column:

    df_service_analytics = df_service_orders.agg(
        count("*").alias("total_service_orders"),
        sum(col(cost_column)).alias("total_service_cost"),
        avg(col(cost_column)).alias("average_service_cost")
    )

else:

    df_service_analytics = df_service_orders.agg(
        count("*").alias("total_service_orders")
    )

# Write Gold table
df_service_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.gold.service_analytics"
    )

print("Gold service analytics created successfully!")

display(df_service_analytics)

Service Orders columns:
['order_id', 'appointment_id', 'assigned_technician_id', 'check_in_time', 'completion_time', 'odometer_reading', 'order_status', 'ingestion_timestamp']
Gold service analytics created successfully!


total_service_orders
10000


In [0]:
# ============================================================
# GOLD LAYER - WARRANTY ANALYTICS
# ============================================================

from pyspark.sql.functions import count, sum, avg, col

# Read Silver warranty claims
df_warranty = spark.table(
    "automotive_warranty.silver.warranty_claims"
)

print("Warranty Claims columns:")
print(df_warranty.columns)

# Find available claim amount column
amount_column = None

for c in [
    "claim_amount",
    "warranty_amount",
    "amount",
    "approved_amount",
    "total_amount",
    "cost"
]:
    if c in df_warranty.columns:
        amount_column = c
        break

# Create warranty analytics
if amount_column:

    df_warranty_analytics = df_warranty.agg(
        count("*").alias("total_warranty_claims"),
        sum(col(amount_column)).alias("total_claim_amount"),
        avg(col(amount_column)).alias("average_claim_amount")
    )

else:

    df_warranty_analytics = df_warranty.agg(
        count("*").alias("total_warranty_claims")
    )

# Write Gold table
df_warranty_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.gold.warranty_analytics"
    )

print("Gold warranty analytics created successfully!")

display(df_warranty_analytics)

Warranty Claims columns:
['claim_id', 'order_id', 'claim_number', 'claim_type', 'submission_date', 'claimed_amount', 'approved_amount', 'claim_status', 'ingestion_timestamp']
Gold warranty analytics created successfully!


total_warranty_claims,total_claim_amount,average_claim_amount
3500,1.7124874E7,4892.821142857143


In [0]:
# ============================================================
# GOLD LAYER - PARTS ANALYTICS
# ============================================================

from pyspark.sql.functions import count, sum, avg, col

# Read Silver parts_used
df_parts_used = spark.table(
    "automotive_warranty.silver.parts_used"
)

print("Parts Used columns:")
print(df_parts_used.columns)

# Find available quantity column
quantity_column = None

for c in ["quantity", "qty", "parts_quantity"]:
    if c in df_parts_used.columns:
        quantity_column = c
        break

# Find available cost column
cost_column = None

for c in ["total_cost", "part_cost", "cost", "amount"]:
    if c in df_parts_used.columns:
        cost_column = c
        break

# Create Parts Analytics
if quantity_column and cost_column:

    df_parts_analytics = df_parts_used.agg(
        count("*").alias("total_parts_used_records"),
        sum(col(quantity_column)).alias("total_parts_quantity"),
        sum(col(cost_column)).alias("total_parts_cost"),
        avg(col(cost_column)).alias("average_part_cost")
    )

elif quantity_column:

    df_parts_analytics = df_parts_used.agg(
        count("*").alias("total_parts_used_records"),
        sum(col(quantity_column)).alias("total_parts_quantity")
    )

elif cost_column:

    df_parts_analytics = df_parts_used.agg(
        count("*").alias("total_parts_used_records"),
        sum(col(cost_column)).alias("total_parts_cost"),
        avg(col(cost_column)).alias("average_part_cost")
    )

else:

    df_parts_analytics = df_parts_used.agg(
        count("*").alias("total_parts_used_records")
    )

# Write Gold table
df_parts_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.gold.parts_analytics"
    )

print("Gold parts analytics created successfully!")

display(df_parts_analytics)

Parts Used columns:
['parts_used_id', 'order_id', 'part_id', 'quantity', 'unit_billed_price', 'total_part_cost', 'ingestion_timestamp']
Gold parts analytics created successfully!


total_parts_used_records,total_parts_quantity
500000,799494


In [0]:
# ============================================================
# GOLD LAYER - FINAL VALIDATION
# ============================================================

gold_tables = [
    "customer_analytics",
    "vehicle_analytics",
    "service_analytics",
    "warranty_analytics",
    "parts_analytics"
]

print("============================================================")
print("GOLD LAYER VALIDATION")
print("============================================================")

for table_name in gold_tables:

    table_full_name = f"automotive_warranty.gold.{table_name}"

    df = spark.table(table_full_name)

    row_count = df.count()

    print(
        f"{table_name:25} | Rows: {row_count:,} | Status: SUCCESS"
    )

print("============================================================")
print("Gold Layer Validation Completed Successfully!")
print("============================================================")

GOLD LAYER VALIDATION
customer_analytics        | Rows: 10,000 | Status: SUCCESS
vehicle_analytics         | Rows: 10,000 | Status: SUCCESS
service_analytics         | Rows: 1 | Status: SUCCESS
warranty_analytics        | Rows: 1 | Status: SUCCESS
parts_analytics           | Rows: 1 | Status: SUCCESS
Gold Layer Validation Completed Successfully!


In [0]:
# ============================================================
# GOLD LAYER - DASHBOARD SUMMARY
# ============================================================

from pyspark.sql.functions import count, sum, avg, col

# Read Gold tables
df_customers = spark.table(
    "automotive_warranty.gold.customer_analytics"
)

df_vehicles = spark.table(
    "automotive_warranty.gold.vehicle_analytics"
)

df_services = spark.table(
    "automotive_warranty.gold.service_analytics"
)

df_warranty = spark.table(
    "automotive_warranty.gold.warranty_analytics"
)

df_parts = spark.table(
    "automotive_warranty.gold.parts_analytics"
)

# Create overall KPI summary
dashboard_summary = spark.createDataFrame([
    (
        df_customers.count(),
        df_vehicles.count(),
        spark.table(
            "automotive_warranty.silver.service_orders"
        ).count(),
        spark.table(
            "automotive_warranty.silver.warranty_claims"
        ).count(),
        spark.table(
            "automotive_warranty.silver.parts_used"
        ).count()
    )
], [
    "total_customers",
    "total_vehicles",
    "total_service_orders",
    "total_warranty_claims",
    "total_parts_used"
])

# Save dashboard summary
dashboard_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.gold.dashboard_summary"
    )

print("Dashboard summary created successfully!")

display(dashboard_summary)

Dashboard summary created successfully!


total_customers,total_vehicles,total_service_orders,total_warranty_claims,total_parts_used
10000,10000,10000,3500,500000
